<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Laboratorios/labo4/Laboratorio_4_visualizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
from IPython.display import HTML, display

display(HTML("""
<style>
#lab-wrapper{
  background: #1e1e24;
  color: #f0f0f0;
  border-radius: 12px;
  padding: 16px 12px;
  box-shadow: 0 4px 16px rgba(0,0,0,0.5);
  border: 1px solid #33333f;
  max-width: 840px;
  margin: 10px auto;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  text-align: center;
  user-select: none;
}

#lab-wrapper h2{
  margin: 4px 0 12px 0;
  color: #ffffff;
  font-size: 20px;
  letter-spacing: 0.5px;
}

.canvas-wrap{
  display: inline-block;
  margin: 6px;
  background: #121216;
  padding: 8px;
  border-radius: 8px;
  border: 1px solid #2a2a35;
}

#lab-wrapper canvas{
  width: 350px;
  height: 350px;
  background: #000;
  border: 1px solid #444;
  border-radius: 4px;
  display: block;
  cursor: crosshair;
  touch-action: none;
}

.label-badge{
  display: inline-block;
  font-weight: 600;
  font-size: 13px;
  margin-bottom: 8px;
  padding: 4px 10px;
  background: rgba(255, 255, 255, 0.08);
  color: #e0e0e0;
  border-radius: 20px;
  border: 1px solid rgba(255, 255, 255, 0.15);
  letter-spacing: 0.3px;
}

.controls-bar{
  margin-top: 10px;
  display: flex;
  flex-wrap: wrap;
  justify-content: center;
  align-items: center;
  gap: 6px;
}

#lab-wrapper button, #lab-wrapper input[type="file"]{
  padding: 6px 12px;
  border-radius: 6px;
  border: 1px solid #4a4a5a;
  background: #2b2b36;
  color: #ffffff;
  font-size: 13px;
  cursor: pointer;
  transition: background 0.15s ease, border-color 0.15s ease;
}

#lab-wrapper button:hover:not(:disabled){
  background: #3b3b4a;
  border-color: #6a6a80;
}

#lab-wrapper button:disabled{
  opacity: 0.4;
  cursor: not-allowed;
}

.range-wrap{
  display: inline-flex;
  align-items: center;
  gap: 8px;
  font-size: 13px;
  color: #ddd;
  background: #2b2b36;
  padding: 4px 10px;
  border-radius: 6px;
  border: 1px solid #4a4a5a;
}

#lab-wrapper input[type="range"]{
  cursor: pointer;
  accent-color: #4a90e2;
}

#lab-wrapper input[type="file"]{
  max-width: 250px;
}

.tool-btn.active, .mode-btn.active{
  background: #2563eb !important;
  border-color: #60a5fa !important;
  color: #ffffff !important;
}

#lab-wrapper input[type="range"]:disabled{
  opacity: 0.4;
  cursor: not-allowed;
}

.filter-btn{
  background: #1e293b !important;
  border-color: #334155 !important;
  font-size: 12px !important;
}
.filter-btn:hover{
  background: #334155 !important;
}
</style>

<div id="lab-wrapper">
  <h2>🔬 Laboratorio Interactivo de Fourier</h2>

  <div style="margin-bottom: 10px;">
    <input type="file" id="file" accept="image/*">
  </div>

  <div class="canvas-wrap">
    <div class="label-badge">🎨 Dominio Espacial (Imagen)</div>
    <canvas id="spatial" width="256" height="256"></canvas>
  </div>

  <div class="canvas-wrap">
    <div class="label-badge">✨ Dominio de Frecuencia (Fourier)</div>
    <canvas id="fourier" width="256" height="256"></canvas>
  </div>

  <div class="controls-bar">
    <button id="pencil" class="mode-btn active">✏️ Dibujar</button>
    <button id="eraser" class="mode-btn">🧹 Borrar</button>
  </div>

  <div class="controls-bar">
    <button id="tool-brush" class="tool-btn active">⚪ Pincel</button>
    <button id="tool-hline" class="tool-btn">↔️ Línea H</button>
    <button id="tool-vline" class="tool-btn">↕️ Línea V</button>
  </div>

  <div class="controls-bar">
    <span style="font-size: 12px; color: #94a3b8; margin-right: 4px;">Filtros:</span>
    <button id="filter-lowpass" class="filter-btn">🔵 Pasa-Bajos</button>
    <button id="filter-highpass" class="filter-btn">🔴 Pasa-Altos</button>
    <button id="filter-denoise" class="filter-btn">🧹 Quitar Rayas</button>
    <button id="filter-pure" class="filter-btn">🌊 Frecuencia Pura</button>
  </div>

  <div class="controls-bar">
    <button id="undo" disabled>↶ Deshacer (0)</button>
    <button id="clear">🗑️ Limpiar Todo</button>
    <button id="save">💾 Guardar Imagen</button>

    <div class="range-wrap">
      <span>Grosor</span>
      <input type="range" id="size" min="1" max="18" value="2">
    </div>
    <div class="range-wrap">
      <span>Intensidad</span>
      <input type="range" id="intensity" min="20" max="200" value="90">
    </div>
  </div>
</div>

<script>
(()=>{
const N = 256;
const N2 = N * N;
const MAX_HISTORY = 25;
const F = document.getElementById("fourier");
const S = document.getElementById("spatial");
const undoBtn = document.getElementById("undo");
const fc = F.getContext("2d");
const sc = S.getContext("2d");

let re = new Float64Array(N2);
let im = new Float64Array(N2);
let spatial = new Float64Array(N2);

// Cache para no recalcular hypot/log1p dos veces por pixel en cada render
const magCache = new Float64Array(N2);

// Registro de píxeles tocados en el trazo actual para evitar sobreacumulación
const strokeVisited = new Uint8Array(N2);

const historyStack = [];
let brush = 2, erase = false, drawing = false, activeCanvas = null;
let lastX = -1, lastY = -1;
let tool = "brush"; // "brush" | "hline" | "vline"
let intensity = 90; // amplitud visual objetivo (0-255)

// ============================================================
// FFT — implementación radix-4 ("fft.js", MIT license, por indutny)
// ============================================================
function FFTLib(size) {
  this.size = size | 0;
  if (this.size <= 1 || (this.size & (this.size - 1)) !== 0)
    throw new Error('FFT size must be a power of two and bigger than 1');
  this._csize = size << 1;
  var table = new Array(this.size * 2);
  for (var i = 0; i < table.length; i += 2) {
    const angle = Math.PI * i / this.size;
    table[i] = Math.cos(angle);
    table[i + 1] = -Math.sin(angle);
  }
  this.table = table;
  var power = 0;
  for (var t = 1; this.size > t; t <<= 1) power++;
  this._width = power % 2 === 0 ? power - 1 : power;
  this._bitrev = new Array(1 << this._width);
  for (var j = 0; j < this._bitrev.length; j++) {
    this._bitrev[j] = 0;
    for (var shift = 0; shift < this._width; shift += 2) {
      var revShift = this._width - shift - 2;
      this._bitrev[j] |= ((j >>> shift) & 3) << revShift;
    }
  }
  this._out = null;
  this._data = null;
  this._inv = 0;
}

FFTLib.prototype.createComplexArray = function createComplexArray() {
  const res = new Array(this._csize);
  for (var i = 0; i < res.length; i++) res[i] = 0;
  return res;
};

FFTLib.prototype.transform = function transform(out, data) {
  if (out === data) throw new Error('Input and output buffers must be different');
  this._out = out; this._data = data; this._inv = 0;
  this._transform4();
  this._out = null; this._data = null;
};

FFTLib.prototype.inverseTransform = function inverseTransform(out, data) {
  if (out === data) throw new Error('Input and output buffers must be different');
  this._out = out; this._data = data; this._inv = 1;
  this._transform4();
  for (var i = 0; i < out.length; i++) out[i] /= this.size;
  this._out = null; this._data = null;
};

FFTLib.prototype._transform4 = function _transform4() {
  var out = this._out;
  var size = this._csize;
  var width = this._width;
  var step = 1 << width;
  var len = (size / step) << 1;
  var outOff, t;
  var bitrev = this._bitrev;
  if (len === 4) {
    for (outOff = 0, t = 0; outOff < size; outOff += len, t++) {
      const off = bitrev[t];
      this._singleTransform2(outOff, off, step);
    }
  } else {
    for (outOff = 0, t = 0; outOff < size; outOff += len, t++) {
      const off = bitrev[t];
      this._singleTransform4(outOff, off, step);
    }
  }
  var inv = this._inv ? -1 : 1;
  var table = this.table;
  for (step >>= 2; step >= 2; step >>= 2) {
    len = (size / step) << 1;
    var quarterLen = len >>> 2;
    for (outOff = 0; outOff < size; outOff += len) {
      var limit = outOff + quarterLen;
      for (var i = outOff, k = 0; i < limit; i += 2, k += step) {
        const A = i;
        const B = A + quarterLen;
        const C = B + quarterLen;
        const D = C + quarterLen;
        const Ar = out[A], Ai = out[A + 1];
        const Br = out[B], Bi = out[B + 1];
        const Cr = out[C], Ci = out[C + 1];
        const Dr = out[D], Di = out[D + 1];
        const MAr = Ar, MAi = Ai;
        const tableBr = table[k], tableBi = inv * table[k + 1];
        const MBr = Br * tableBr - Bi * tableBi;
        const MBi = Br * tableBi + Bi * tableBr;
        const tableCr = table[2 * k], tableCi = inv * table[2 * k + 1];
        const MCr = Cr * tableCr - Ci * tableCi;
        const MCi = Cr * tableCi + Ci * tableCr;
        const tableDr = table[3 * k], tableDi = inv * table[3 * k + 1];
        const MDr = Dr * tableDr - Di * tableDi;
        const MDi = Dr * tableDi + Di * tableDr;
        const T0r = MAr + MCr, T0i = MAi + MCi;
        const T1r = MAr - MCr, T1i = MAi - MCi;
        const T2r = MBr + MDr, T2i = MBi + MDi;
        const T3r = inv * (MBr - MDr), T3i = inv * (MBi - MDi);
        const FAr = T0r + T2r, FAi = T0i + T2i;
        const FCr = T0r - T2r, FCi = T0i - T2i;
        const FBr = T1r + T3i, FBi = T1i - T3r;
        const FDr = T1r - T3i, FDi = T1i + T3r;
        out[A] = FAr; out[A + 1] = FAi;
        out[B] = FBr; out[B + 1] = FBi;
        out[C] = FCr; out[C + 1] = FCi;
        out[D] = FDr; out[D + 1] = FDi;
      }
    }
  }
};

FFTLib.prototype._singleTransform2 = function _singleTransform2(outOff, off, step) {
  const out = this._out, data = this._data;
  const evenR = data[off], evenI = data[off + 1];
  const oddR = data[off + step], oddI = data[off + step + 1];
  out[outOff] = evenR + oddR; out[outOff + 1] = evenI + oddI;
  out[outOff + 2] = evenR - oddR; out[outOff + 3] = evenI - oddI;
};

FFTLib.prototype._singleTransform4 = function _singleTransform4(outOff, off, step) {
  const out = this._out, data = this._data;
  const inv = this._inv ? -1 : 1;
  const step2 = step * 2, step3 = step * 3;
  const Ar = data[off], Ai = data[off + 1];
  const Br = data[off + step], Bi = data[off + step + 1];
  const Cr = data[off + step2], Ci = data[off + step2 + 1];
  const Dr = data[off + step3], Di = data[off + step3 + 1];
  const T0r = Ar + Cr, T0i = Ai + Ci;
  const T1r = Ar - Cr, T1i = Ai - Ci;
  const T2r = Br + Dr, T2i = Bi + Di;
  const T3r = inv * (Br - Dr), T3i = inv * (Bi - Di);
  const FAr = T0r + T2r, FAi = T0i + T2i;
  const FBr = T1r + T3i, FBi = T1i - T3r;
  const FCr = T0r - T2r, FCi = T0i - T2i;
  const FDr = T1r - T3i, FDi = T1i + T3r;
  out[outOff] = FAr; out[outOff + 1] = FAi;
  out[outOff + 2] = FBr; out[outOff + 3] = FBi;
  out[outOff + 4] = FCr; out[outOff + 5] = FCi;
  out[outOff + 6] = FDr; out[outOff + 7] = FDi;
};

const fftInstance = new FFTLib(N);
const fftRowIn = fftInstance.createComplexArray();
const fftRowOut = fftInstance.createComplexArray();

function fft2(r, q, inverse) {
  for (let y = 0; y < N; y++) {
    for (let x = 0; x < N; x++) {
      fftRowIn[2 * x] = r[y * N + x];
      fftRowIn[2 * x + 1] = q[y * N + x];
    }
    if (inverse) fftInstance.inverseTransform(fftRowOut, fftRowIn);
    else fftInstance.transform(fftRowOut, fftRowIn);
    for (let x = 0; x < N; x++) {
      r[y * N + x] = fftRowOut[2 * x];
      q[y * N + x] = fftRowOut[2 * x + 1];
    }
  }
  for (let x = 0; x < N; x++) {
    for (let y = 0; y < N; y++) {
      fftRowIn[2 * y] = r[y * N + x];
      fftRowIn[2 * y + 1] = q[y * N + x];
    }
    if (inverse) fftInstance.inverseTransform(fftRowOut, fftRowIn);
    else fftInstance.transform(fftRowOut, fftRowIn);
    for (let y = 0; y < N; y++) {
      r[y * N + x] = fftRowOut[2 * y];
      q[y * N + x] = fftRowOut[2 * y + 1];
    }
  }
}

// --- Render del espectro de Fourier ---
function renderFourier() {
  let maxVal = 0;

  for (let i = 0; i < N2; i++) {
    let m = Math.log1p(Math.hypot(re[i], im[i]));
    magCache[i] = m;
    if (i !== 0 && m > maxVal) maxVal = m; // índice 0 = DC, se excluye para normalizar
  }
  if (maxVal === 0 && magCache[0] > 0) maxVal = magCache[0];

  let z = fc.createImageData(N, N);
  for (let y = 0; y < N; y++) {
    for (let x = 0; x < N; x++) {
      let X = (x + N / 2) & 255, Y = (y + N / 2) & 255, i = Y * N + X;
      let v = maxVal > 0 ? Math.min(255, Math.round(magCache[i] * 255 / maxVal)) : 0;
      let p = (y * N + x) * 4;
      z.data[p] = v;
      z.data[p + 1] = v;
      z.data[p + 2] = v;
      z.data[p + 3] = 255;
    }
  }
  fc.putImageData(z, 0, 0);
}

// --- Render del dominio espacial ---
// Corrección pedagógica fundamental:
// Cuando no hay un componente DC significativo (es una frecuencia pura sobre
// fondo negro), los valores en el espacio oscilan entre -A y +A (media 0).
// En vez de recortar la mitad negativa a cero con Math.max(0, ...), centramos
// la oscilación en gris medio (128) para que el coseno se aprecie completo
// (crestas claras y valles oscuros). Si hay una imagen cargada con brillo positivo,
// se muestra directamente en el rango normal [0, 255].
function renderSpatial() {
  let z = sc.createImageData(N, N);
  let meanDC = re[0] / N2;
  let minV = Infinity, maxV = -Infinity;

  for (let i = 0; i < N2; i++) {
    if (spatial[i] < minV) minV = spatial[i];
    if (spatial[i] > maxV) maxV = spatial[i];
  }

  let isPureAC = (Math.abs(meanDC) < 2.0 && minV < -0.5);
  let peak = Math.max(Math.abs(minV), Math.abs(maxV));
  let scale = (isPureAC && peak > 127) ? (127 / peak) : 1.0;

  for (let i = 0; i < N2; i++) {
    let val = isPureAC ? (128 + spatial[i] * scale) : spatial[i];
    let v = Math.min(255, Math.max(0, Math.round(val)));
    let p = i * 4;
    z.data[p] = v;
    z.data[p + 1] = v;
    z.data[p + 2] = v;
    z.data[p + 3] = 255;
  }
  sc.putImageData(z, 0, 0);
}

function updateSpatialFromFourier() {
  let a = new Float64Array(re), b = new Float64Array(im);
  fft2(a, b, true);
  for (let i = 0; i < N2; i++) spatial[i] = a[i];
  renderSpatial();
}

function updateFourierFromSpatial() {
  for (let i = 0; i < N2; i++) {
    re[i] = spatial[i];
    im[i] = 0;
  }
  fft2(re, im, false);
  renderFourier();
}

function updateUndoButton() {
  undoBtn.disabled = historyStack.length === 0;
  undoBtn.innerText = `↶ Deshacer (${historyStack.length})`;
}

function saveState() {
  if (historyStack.length >= MAX_HISTORY) {
    historyStack.shift();
  }
  historyStack.push({
    spatial: new Float64Array(spatial),
    re: new Float64Array(re),
    im: new Float64Array(im)
  });
  updateUndoButton();
}

// Escribe o borra frecuencias manteniendo la simetría hermítica exacta.
// Utiliza `strokeVisited` para que el movimiento del mouse no acumule
// energía infinitas veces sobre los mismos píxeles.
function setPixel(x, y, canvas, strength) {
  if (x < 0 || x >= N || y < 0 || y >= N) return;
  if (canvas === S) {
    spatial[y * N + x] = erase ? 0 : 255;
  } else {
    let X = (x + N / 2) & 255, Y = (y + N / 2) & 255;
    let i1 = Y * N + X;
    let symX = (N - X) & 255, symY = (N - Y) & 255;
    let i2 = symY * N + symX;
    let selfConjugate = (i1 === i2);

    if (erase) {
      re[i1] = im[i1] = 0;
      if (!selfConjugate) { re[i2] = im[i2] = 0; }
    } else {
      if (!strokeVisited[i1]) {
        re[i1] += strength;
        strokeVisited[i1] = 1;
        if (!selfConjugate) {
          re[i2] += strength;
          strokeVisited[i2] = 1;
        }
      }
    }
  }
}

function countBrushPoints(r) {
  let c = 0;
  for (let dy = -r; dy <= r; dy++) {
    for (let dx = -r; dx <= r; dx++) {
      if (dx * dx + dy * dy <= r * r) c++;
    }
  }
  return Math.max(1, c);
}

function applyBrushAt(cx, cy, canvas) {
  const pts = countBrushPoints(brush);
  const strength = (canvas === F) ? (intensity * N2) / (2 * pts) : 0;
  for (let dy = -brush; dy <= brush; dy++) {
    for (let dx = -brush; dx <= brush; dx++) {
      if (dx * dx + dy * dy <= brush * brush) {
        setPixel(cx + dx, cy + dy, canvas, strength);
      }
    }
  }
}

function applyHLineAt(cy, canvas) {
  if (canvas === S) {
    for (let x = 0; x < N; x++) spatial[cy * N + x] = erase ? 0 : 255;
  } else {
    const strength = (intensity * N) / 2;
    for (let x = 0; x < N; x++) setPixel(x, cy, canvas, strength);
  }
}

function applyVLineAt(cx, canvas) {
  if (canvas === S) {
    for (let y = 0; y < N; y++) spatial[y * N + cx] = erase ? 0 : 255;
  } else {
    const strength = (intensity * N) / 2;
    for (let y = 0; y < N; y++) setPixel(cx, y, canvas, strength);
  }
}

function applyToolAt(cx, cy, canvas) {
  if (tool === "hline") applyHLineAt(cy, canvas);
  else if (tool === "vline") applyVLineAt(cx, canvas);
  else applyBrushAt(cx, cy, canvas);
}

function interpolateAndDraw(x1, y1, x2, y2, canvas) {
  let dist = Math.hypot(x2 - x1, y2 - y1);
  let steps = Math.max(1, Math.ceil(dist / 2));
  for (let s = 0; s <= steps; s++) {
    let t = s / steps;
    let cx = Math.round(x1 + (x2 - x1) * t);
    let cy = Math.round(y1 + (y2 - y1) * t);
    applyToolAt(cx, cy, canvas);
  }
}

function handlePointer(e, isDown = false) {
  if (!drawing && !isDown) return;
  let rect = activeCanvas.getBoundingClientRect();
  let x = Math.floor((e.clientX - rect.left) / rect.width * N);
  let y = Math.floor((e.clientY - rect.top) / rect.height * N);

  if (isDown) {
    lastX = x; lastY = y;
    applyToolAt(x, y, activeCanvas);
  } else {
    interpolateAndDraw(lastX, lastY, x, y, activeCanvas);
    lastX = x; lastY = y;
  }

  if (activeCanvas === S) {
    renderSpatial();
    updateFourierFromSpatial();
  } else {
    renderFourier();
    updateSpatialFromFourier();
  }
}

function startStroke(e, canvas) {
  strokeVisited.fill(0);
  saveState();
  drawing = true;
  activeCanvas = canvas;
  handlePointer(e, true);
}

S.onpointerdown = e => startStroke(e, S);
F.onpointerdown = e => startStroke(e, F);

window.addEventListener("pointermove", e => { if (drawing) handlePointer(e, false); });
window.addEventListener("pointerup", () => { drawing = false; activeCanvas = null; });

function loadImage(file) {
  if (!file) return;
  let img = new Image(), u = URL.createObjectURL(file);
  img.onload = () => {
    let c = document.createElement("canvas"), x = c.getContext("2d");
    c.width = c.height = N;
    x.drawImage(img, 0, 0, N, N);
    let d = x.getImageData(0, 0, N, N).data;
    saveState();
    for (let i = 0; i < N2; i++) {
      spatial[i] = 0.299 * d[4 * i] + 0.587 * d[4 * i + 1] + 0.114 * d[4 * i + 2];
    }
    renderSpatial();
    updateFourierFromSpatial();
    URL.revokeObjectURL(u);
  };
  img.src = u;
}

document.getElementById("file").onchange = e => loadImage(e.target.files[0]);
document.getElementById("size").oninput = e => brush = +e.target.value;
document.getElementById("intensity").oninput = e => intensity = +e.target.value;

const pencilBtn = document.getElementById("pencil");
const eraserBtn = document.getElementById("eraser");

pencilBtn.onclick = () => {
  erase = false;
  pencilBtn.classList.add("active");
  eraserBtn.classList.remove("active");
};
eraserBtn.onclick = () => {
  erase = true;
  eraserBtn.classList.add("active");
  pencilBtn.classList.remove("active");
};

const toolButtons = {
  brush: document.getElementById("tool-brush"),
  hline: document.getElementById("tool-hline"),
  vline: document.getElementById("tool-vline"),
};
const sizeSlider = document.getElementById("size");

function selectTool(name) {
  tool = name;
  Object.entries(toolButtons).forEach(([key, btn]) => {
    btn.classList.toggle("active", key === name);
  });
  sizeSlider.disabled = (name !== "brush");
}

toolButtons.brush.onclick = () => selectTool("brush");
toolButtons.hline.onclick = () => selectTool("hline");
toolButtons.vline.onclick = () => selectTool("vline");

// Filtros rápidos interactivos
document.getElementById("filter-lowpass").onclick = () => {
  saveState();
  const cutoff2 = 35 * 35;
  for (let y = 0; y < N; y++) {
    let dy = y < 128 ? y : y - N;
    for (let x = 0; x < N; x++) {
      let dx = x < 128 ? x : x - N;
      if (dx * dx + dy * dy > cutoff2) {
        let i = y * N + x;
        re[i] = 0; im[i] = 0;
      }
    }
  }
  renderFourier();
  updateSpatialFromFourier();
};

document.getElementById("filter-highpass").onclick = () => {
  saveState();
  const cutoff2 = 20 * 20;
  for (let y = 0; y < N; y++) {
    let dy = y < 128 ? y : y - N;
    for (let x = 0; x < N; x++) {
      let dx = x < 128 ? x : x - N;
      if (dx * dx + dy * dy <= cutoff2) {
        let i = y * N + x;
        re[i] = 0; im[i] = 0;
      }
    }
  }
  renderFourier();
  updateSpatialFromFourier();
};

document.getElementById("filter-denoise").onclick = () => {
  saveState();
  // Filtro Notch: elimina ruido periódico en los ejes horizontal y vertical (conservando el DC)
  for (let k = 1; k < N; k++) {
    re[k] = 0; im[k] = 0;
    re[k * N] = 0; im[k * N] = 0;
  }
  renderFourier();
  updateSpatialFromFourier();
};

document.getElementById("filter-pure").onclick = () => {
  saveState();
  re.fill(0); im.fill(0);
  strokeVisited.fill(0);
  const strength = (intensity * N2) / 2;
  setPixel(128 + 20, 128, F, strength);
  renderFourier();
  updateSpatialFromFourier();
};

undoBtn.onclick = () => {
  if (historyStack.length > 0) {
    const prevState = historyStack.pop();
    spatial.set(prevState.spatial);
    re.set(prevState.re);
    im.set(prevState.im);
    renderSpatial();
    renderFourier();
    updateUndoButton();
  }
};

document.getElementById("clear").onclick = () => {
  saveState();
  spatial.fill(0);
  re.fill(0);
  im.fill(0);
  renderSpatial();
  renderFourier();
};

document.getElementById("save").onclick = () => {
  let a = document.createElement("a");
  a.download = "resultado_espacial.png";
  a.href = S.toDataURL("image/png");
  a.click();
};

updateUndoButton();

})();
</script>
"""))